# 2026可审计断点续跑（修复10月30日阻断）

先停止旧Notebook⑤，保留所有文件。按1→6。2020—2025不重跑，原参数不变；第4格在金融计算前校验并解析尾段36份逐笔。第5格只迁移经过严格旧代码、数据、信号、配置、已用SHA、账户与完整历史权益检查的断点；不强行忽略指纹。已完成账户同样核验后复用。

已用真实BTC CANCEL旧断点验证恢复到北京时间10月30日08:00，435360个分钟权益与账户状态保持。其他组由Colab逐一核验；没有旧断点的未运行阶段才正常从年初起算。

## 1. 环境

In [ ]:
import sys,os,subprocess,shutil,platform
assert sys.version_info >= (3,12), '请使用Python3.12或更高的Colab运行时'
subprocess.run([sys.executable,'-m','pip','-q','install','numpy==2.3.5','pandas==3.0.1','psutil==7.0.0'],check=True)
import psutil
print('Python:',sys.version.split()[0], 'CPU:',os.cpu_count(),platform.machine())
print('RAM available/total GB:',round(psutil.virtual_memory().available/2**30,1),round(psutil.virtual_memory().total/2**30,1))
print('Local disk free GB:',round(shutil.disk_usage('/content').free/2**30,1))
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True).stdout)
else: print('GPU: none. This financial replay uses CPU.')

## 2. 锁定代码
读取私有GitHub仓库需要Colab Secret GITHUB_TOKEN，或隐藏输入，不保存或打印凭据。

In [ ]:
from pathlib import Path
import tempfile,getpass,json
CODE_COMMIT = '4094d4cc9941db51b014dff63f6c51a4b37fb2a5'
REPO = Path('/content/er-v2-2026-revision-code-'+CODE_COMMIT[:7])
if (REPO/'.git').exists():
    assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()==CODE_COMMIT, '目录提交不一致；请使用新目录，不覆盖旧任务'
else:
    if REPO.exists() and any(REPO.iterdir()): raise RuntimeError('目标目录非空；换一个新目录')
    try:
        from google.colab import userdata
        token=userdata.get('GITHUB_TOKEN')
    except Exception:
        token=getpass.getpass('GitHub private repository token (hidden): ')
    if not token: raise RuntimeError('缺少私有仓库读取凭据')
    env=os.environ.copy();env['COLAB_GH_TOKEN']=token;env['GIT_TERMINAL_PROMPT']='0'
    with tempfile.TemporaryDirectory() as tmp:
        ask=Path(tmp)/'askpass.py'
        ask.write_text('#!/usr/bin/env python3\nimport os,sys\nprint("x-access-token" if "username" in sys.argv[1].lower() else os.environ["COLAB_GH_TOKEN"])\n')
        ask.chmod(0o700);env['GIT_ASKPASS']=str(ask)
        try:
            REPO.mkdir(parents=True,exist_ok=True)
            for cmd in [['git','init','-q'],['git','remote','add','origin','https://github.com/HITGoose/btc-eth-backtest.git'],['git','-c','credential.helper=','fetch','-q','--depth=1','origin',CODE_COMMIT],['git','checkout','-q','--detach','FETCH_HEAD']]:
                subprocess.run(cmd,cwd=REPO,env=env,check=True,capture_output=True)
        finally:
            env.pop('COLAB_GH_TOKEN',None);del token
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()==CODE_COMMIT
os.environ['BTC_REPO_ROOT']=str(REPO)
ENTRY=REPO/'outputs/er_v2_2026_revision'
print('Locked implementation:',CODE_COMMIT)
print('Repository:',REPO)

## 3. Drive及恢复目录
旧2026_OFFICIAL_REVISION_ae8fbb1只读接入，不覆盖。新结果仍在原共享目录下的2026_AUDITED_RESUME子目录。若当前目录已有本修复版结果则恢复其断点。

In [ ]:
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir(): drive.mount('/content/drive')
DRIVE=Path('/content/drive/MyDrive')
SOURCE=DRIVE/'btc_eth_er_v2_retests/6a26255'
assert SOURCE.is_dir(), '找不到原Drive结果目录'
DRIVE_DATA=DRIVE/'btc_eth_market_cache'
PREVIOUS=SOURCE/'2026_OFFICIAL_REVISION_ae8fbb1'
assert (PREVIOUS/'data_revision.json').is_file(), '原2026补跑清单缺失，不能核验断点'
DRIVE_RESULTS=SOURCE/('2026_AUDITED_RESUME_'+CODE_COMMIT[:7])
DATA=Path('/content/btc_eth_market_cache_revision')
RESULTS=Path('/content/er_v2_2026_revision_results_'+CODE_COMMIT[:7])
def restore(source,target,prior_only=False):
    if not source.exists(): return
    for p in source.rglob('*'):
        if not p.is_file() or p.name.endswith(('.copying','.tmp','.part','.restoring')): continue
        if prior_only and (p.suffix not in ['.json','.npz'] or 'smoke' in p.parts): continue
        dest=target/p.relative_to(source)
        if dest.exists() and dest.stat().st_mtime>=p.stat().st_mtime and dest.stat().st_size==p.stat().st_size: continue
        dest.parent.mkdir(parents=True,exist_ok=True)
        temp=dest.with_suffix(dest.suffix+'.restoring');shutil.copy2(p,temp);temp.replace(dest)
DATA.mkdir(parents=True,exist_ok=True);RESULTS.mkdir(parents=True,exist_ok=True)
restore(DRIVE_DATA,DATA);restore(DRIVE_RESULTS,RESULTS)
os.environ.update(BTC_REPO_ROOT=str(REPO),BTC_DATA_ROOT=str(DATA),BTC_OUTPUT_ROOT=str(RESULTS),BTC_DRIVE_DATA=str(DRIVE_DATA),BTC_DRIVE_RESULTS=str(DRIVE_RESULTS),PYTHONUNBUFFERED='1')
print('旧结果只读：',SOURCE)
print('旧2026断点只读接入：',PREVIOUS)
print('本次结果和断点：',DRIVE_RESULTS)
print('尚未启动账户回放。')


## 4. 完整数据预检（尚不启动账户）
核验训练冻结与旧ae8fbb1源码指纹；下载并用原逐笔解析器检查10月29日至11月15日的两币共36份档案。数据再次变化、字段或ID错误均拦截，不继续计算。关键档案按SHA备份Drive。已验证的文件下次跳过重复解析。核验旧已完成结果是否可复用。首次此格需要下载与解析时间，页面会显示逐日进度。

In [ ]:
BASE_CMD=[sys.executable,'-u',str(ENTRY/'run.py'),'--source',str(SOURCE),'--previous',str(PREVIOUS)]
subprocess.run(BASE_CMD+['--prepare-only'],cwd=REPO,check=True)
print('准备完成，尚未运行2026账户。下一格才开始。')


## 5. 迁移兼容断点并续算
界面显示FOUND_PREVIOUS_CHECKPOINT_VERIFY_FIRST，审计后显示AUDITED_RESUME／RESUME日期。原代码/数据指纹先严格匹配，再迁移完整状态至新版本；不覆盖旧断点。已完成兼容结果跳过。未曾运行的阶段正常从年初起算。真正失败会显示错误并停止后续阶段，不再让其他阶段无效空转。断线后同本Notebook 1→4再5恢复。

In [ ]:
import signal
process=subprocess.Popen(BASE_CMD,cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,start_new_session=True)
try:
    for line in process.stdout:print(line,end='',flush=True)
    if process.wait()!=0:raise RuntimeError('运行中断或阻断，查看错误；已有结果和断点保留。')
except KeyboardInterrupt:
    try:os.killpg(process.pid,signal.SIGTERM)
    except ProcessLookupError:pass
    process.wait();raise
print('流程结束。查看incomplete.csv确认是否全部完成。')


## 6. 只读汇总
旧2020—2025＋本次统一尾段数据下核验复用/恢复后的2026结果。修正方向标签归因，不改变财务逻辑；未完成不填零。结果保留原共享目录子目录，不用再上传大ZIP。

In [ ]:
subprocess.run(BASE_CMD+['--report-only'],cwd=REPO,check=True)
print((RESULTS/'REPORT.md').read_text())
print('结果目录：',DRIVE_RESULTS)
print('把这个子目录路径告诉我即可，我可通过原共享链接读取。')


## 可选：单独下载CSV（不重跑）

In [ ]:
from google.colab import files
FILE='annual.csv' # 或 incomplete.csv、trade_matching.csv、REPORT.md、data_revision.json
p=RESULTS/FILE;assert p.is_file(), '文件尚未生成'
files.download(str(p))
